⚠️ **Missing input:** `LVVI3_activities_inference_postprocessing.csv` no longer exists in this project — it lived in the deleted `LVVI3_YouTube` project and was lost in the reorganization. The cell below must NOT be run until this LVVI3 activity/sub_activity mapping file is reconstructed.

# 04 YouTube activity labeling (LEGACY)

Extracts per-shot labels from the old Qwen3-VL run, maps sub_activity names to the LVVI3 taxonomy. Superseded by the clef-model pipeline in `scripts/YouTube/multimodal_analysis/`. `sub_activity` is kept only at the preprocessing level.

**Inputs:**
- `../../data/nbt_videos_in_state_owned_lands.csv` — NBT videos on state-owned lands
- `../../../data/YouTube/nbt_videos_in_state_owned_lands_cleaned.csv` — cleaned subset used for labeling
- `../../../data/_archive/old_llm_subactivity/youtube/inference_json_output/` — old Qwen3-VL per-shot activity JSONs (archived)
- `../../../data/_archive/old_llm_subactivity/youtube/accompany_inference_json_output/` — old Qwen3-VL accompany-type JSONs (archived)
- `../../../data/YouTube/LVVI3_activities_inference_postprocessing.csv` — LVVI3 activity/sub_activity mapping (**missing, see note in the notebook**)

**Outputs:**
- `../../../data/_archive/old_llm_subactivity/youtube/YouTube_inferenced.csv` — per-video inferred activity labels (archived)

**Pipeline position:** step 04 of `scripts/YouTube/data_preprocessing/` — legacy, superseded by the Clef pipeline in `scripts/YouTube/multimodal_analysis/`.

# Preparations

In [1]:
import pandas as pd
import json
from pathlib import Path

# Clean channels and videos

## Read YouTube data

In [2]:
df = pd.read_csv('../../data/nbt_videos_in_state_owned_lands.csv')
df.shape

(6222, 11)

## Identify 4K landscape videos

In [3]:
# To remove channels that have more than 5 videos with keywords "4k" or "drone"
ls_landscape_keywords = ['4k', 'drone']
pattern = '|'.join(ls_landscape_keywords)
mask = df['title'].str.contains(pattern, case=False, na=False) | df['description'].str.contains(pattern, case=False, na=False) | df['tags'].str.contains(pattern, case=False, na=False)
df_landscape = df[mask]
ls_channels_landscape = df_landscape['channel_id'].unique().tolist()
len(ls_channels_landscape)

292

In [64]:
# To keep channels that have less than 5 videos with keywords "4k" or "drone"
channel_counts = df_landscape.groupby(['channel_id']).size().sort_values(ascending=False)
ls_channels_landscape_to_keep = channel_counts[channel_counts < 5].index.unique().tolist()
len(ls_channels_landscape_to_keep)

270

In [ ]:
# To keep channels that are manually identified as relevant to the research topic
ls_channels_landscape_to_keep.extend(['UC6G2cmWrqeAqyhy2reO72Xg', 'UC16NdAq6BIn7fy7sX2JTWzw', 'UCZ2GLsGoD9oqvBClzeseqKg', 'UCt529HCZxcbVtpvbNXsxUhA',
                            'UCSNQ6urg5wx1VCtNjiyaxqA', 'UChBPAnbOE1bTbvEp-dttZXA', 'UCDH_XEf4KZeQflV8DBzXWAw', 'UC9Au1qH3EhKjX31pYKlfz4A', 'UCcEyJ3LOAHTEYCk2s3l7ydQ',
                            'UC9Q0cWgt_Oa0pM_m_bchLjw', 'UCP_hsjEKYt9P2SwFUeOWlZg', 'UCrJygwIZvCQXOgYrwNlkqxA'])
len(ls_channels_landscape_to_keep)

282

In [71]:
ls_channels_landscape_to_remove = list(set(ls_channels_landscape) - set(ls_channels_landscape_to_keep))
ls_video_ids_landscape = df[df['channel_id'].isin(ls_channels_landscape_to_remove)]['video_id'].tolist()
len(ls_video_ids_landscape)

138

## Identify videos by channels relevant with Santa Claus

In [72]:
df_santa = df[df['channel_title'].str.contains('santa', case=False, na=False)]
ls_video_ids_santa = df_santa['video_id'].tolist()
len(ls_video_ids_santa)

45

## Identify videos with music topics

In [73]:
ls_music_keywords = ['electronic_music', 'rock_music', 'christian_music', 'Classical_music', 'pop_music',
                     'independent_music']
pattern = '|'.join(ls_music_keywords)
mask = df['topics'].str.contains(pattern, case=False, na=False)
df_music = df[mask]
ls_video_ids_music = df_music['video_id'].tolist()
len(ls_video_ids_music)

36

## Remove identified channels and videos

In [74]:
ls_video_ids_to_remove = list(set(ls_video_ids_landscape) | set(ls_video_ids_santa) | set(ls_video_ids_music))
len(ls_video_ids_to_remove)

216

In [81]:
df_cleaned = df[~df['video_id'].isin(ls_video_ids_to_remove)]
df_cleaned.shape

(6006, 11)

## Save cleaned data

In [3]:
# df_cleaned.to_csv('../../../data/YouTube/nbt_videos_in_state_owned_lands_cleaned.csv', index=False)
df_cleaned = pd.read_csv('../../../data/YouTube/nbt_videos_in_state_owned_lands_cleaned.csv')
ls_video_id = df_cleaned.video_id.tolist()

# Clean activities

## Read LVVI3 activities for postprocessing

In [5]:
df_activities_post_origin = pd.read_csv('../../../data/YouTube/LVVI3_activities_inference_postprocessing.csv')
df_activities_post = df_activities_post_origin[df_activities_post_origin['type']=='survey_activity']
ls_activities_post = df_activities_post['activity'].unique().tolist()
ls_subactivities_post = df_activities_post['sub_activity'].unique().tolist()
ls_activities_post = list(set(ls_activities_post + ls_subactivities_post))
print(ls_activities_post)

['Group exercise e.g. gymnastics or dance in a park or in nature', 'Staying overnight in a tent, lodge, etc. in natural terrain', 'Boating', 'Flying a kite', 'Cross-country skiing', 'Picking herbs, flowers or ornamental plants in nature', 'Canoeing', 'Deer hunting', 'Hiking', 'Diving', 'Cycling', 'Other nature observation', 'Nordic walking', 'Game management', 'Geocaching', 'Camping', 'Rock or mountain climbing', 'Small game hunting', 'Rowing boat', 'Skating', 'Staying in a summer cottage, farm, holiday home', 'Fuelwood procurement, collection of twigs, branches and trees for domestic use', 'Plant', 'Spotting', 'ATV riding', 'Kayaking', 'Swimming', 'Downhill skiing', 'Surfing', 'Nature observation', 'Walking', 'Beach games', 'Forestry work such as planting, nursery management, etc.', 'Sledding', 'Snowmobiling', 'Snorkelling', 'Outdoor activities with children', 'Off-road motorcycling', 'Water sports', 'Four-wheel-drive off-road driving', 'Outdoor event or competition in nature', 'Mushr

## Read inferenced activities from JSON files

In [11]:
# Extract activities from all JSON files and create a DataFrame
def extract_activities(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    video_id = data.get("video_id")
    rows = []

    for shot in data.get("shots", []):
        for activity in shot.get("activities", []):
            name = activity.get("name", "").strip()
            name = name[:1].upper() + name[1:].lower() if name else ""

            act_type = activity.get("type", "").strip()

            rows.append({
                "video_id": video_id,
                "activity": name,
                "type": act_type
            })

    return rows

all_rows = []
json_folder = Path("../../../data/_archive/old_llm_subactivity/youtube/inference_json_output")

for file in json_folder.glob("*.json"):
    all_rows.extend(extract_activities(file))

df_activities = pd.DataFrame(all_rows)
df_activities = df_activities[df_activities['video_id'].isin(ls_video_id)]
df_activities = df_activities.drop_duplicates().reset_index(drop=True)
df_activities.shape

(9419, 3)

## Match with LVVI3 activities

In [112]:
# Remove non-activities
non_activities = [
    "Ruskaretket", "Cooking", "Hanging around with a hammock",
    "Building a fire", "Cooking chores", "Fire making",
    "Drinking beer", "Electric fatbike rental", "Driving",
    "Backpacking", "Lounging or sightseeing on the beach",
    "Meeting animals in a wildlife park", "Snow castle visit",
    "Staying in a snow hotel", "Visiting the snow castle",
    "Visiting snow castle", "Meeting the bear",
    "Motorised outdoor activities", "Survivalism",
    "Snow angels", "Making snow angels", "Building the snowman",
    "Sunbathing, lounging or sightseeing on the beach",
    "Urban exploration (urbex)", "Urban exploration",
    "Sightseeing", "Ruska", "Repairing equipment", "Resting", "Roadtripfinland",
    "Pinturit (fishing with float or lure)", "Building an igloo",
    "Pyykinpesu", "Pyykinpesua", "Ziplining", "Jumping",
    "Gondola ride", "Riding a gondola", "Driving in snow",
    "Shopping", "Shopping for snowboots", "Eating",
    "Underwater drone operation", "Frontflip", "Snow removal",
    "Sleeping", "Experimenting with hot water turning to snow",
    "Fireside relaxation", "Shelter bauen", "Feuer machen",
    "Boat travel", "Face wall", "Day trip to a national park",
    "Parachuting", "Off-grid communications",
    "Off grid ham radio operations", "Off-grid ham radio operations",
    "Collecting snow in the kettle so that we can melt it for coffee, tea, or to make food",
    "Campfire preparation", "Guldvaskning", "Gulvaskning",
    "Gulgrävning", "Dancing", "Lagerfeuer", "Visiting santa's house",
    "Visiting santa claus village", "Gold dredging", "Meeting animals in nature", "Meeting the bear billy",
    "Having lunch in a traditional lappish style", "Cooking over a wood stove", "Revontulet (wildfire / night fire)",
    "Water crossing", "Toasting marshmallows", "Toast marshmallows", "Preparing for outdoor activity", "Reading a book",
    "Relaxing", "Snow activities", "River crossing", "Night exploration in forest", "Night hiking or trekking in forest",
    "Puiden pilkkomista", "Visiting the arcticum museum", "Visiting a museum", "Visiting the village of santa claus",
    "Floating", "Icebreaker ship tour", "Floating in icy arctic sea", "Floating in frozen arctic sea",
    "Visiting engine room", "Meeting captain", "Floating in icy sea", "Skicross", "Staying in a glass igloo",
    "Päiväretki tunturi-suivakolle", "Päiväretki (day hike)", "Päiväretki", "Kahlaamalla", "Feeding", "Bonfire",
    "Rolling in the snow", "Staying in an igloo", "Sliding", "Wandertouren",
    "Ice floating", "Feeding animals", "Husky dogs", "Outdoor cooking", "Setting up the ultamid-tent", "Forest bathing",
    "Snow village visit", "Ice-hotel visit", "Roadtrip", "Water collection", "Water procurement", "Staying in a cabin",
    "Fastpacking", "Ice restaurant dinner", "Lepotauko hammaskurun autiotuvan terassilla lapin erämaassa", "Keilailua",
    
]

df_activities = df_activities[~df_activities["activity"].isin(non_activities)]

In [113]:
# Rename inferenced activities to match with the LVVI3 activity list
df_activities["activity"] = df_activities["activity"].replace("Picnics", "Picnicking")
df_activities["activity"] = df_activities["activity"].replace("Off-piste skiing", "Off-piste skiing or snowboarding off slopes")
df_activities["activity"] = df_activities["activity"].replace("Sunbathing or lounging elsewhere in nature", "Sunbathing")
df_activities["activity"] = df_activities["activity"].replace("Powder skiing", "Skiing")
df_activities["activity"] = df_activities["activity"].replace("Freeriding", "Skiing")
df_activities["activity"] = df_activities["activity"].replace("Trail running", "Running")
df_activities["activity"] = df_activities["activity"].replace("Trailrunning", "Running")
df_activities["activity"] = df_activities["activity"].replace("Birding", "Bird watching")
df_activities["activity"] = df_activities["activity"].replace("Splitboarding", "Snowboarding")
df_activities["activity"] = df_activities["activity"].replace("Tobogganing", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Reno sledding", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Motorsledding", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Sledding, tobogganing", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Sleigh rides", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Atv riding", "ATV riding")
df_activities["activity"] = df_activities["activity"].replace("Fishing with line, hook etc.", "Fishing")
df_activities["activity"] = df_activities["activity"].replace("Drone flying", "Flying a drone")
df_activities["activity"] = df_activities["activity"].replace("Drone", "Flying a drone")
df_activities["activity"] = df_activities["activity"].replace("Treasure hunting, geocaching", "Treasure hunting")
df_activities["activity"] = df_activities["activity"].replace("Snowshoe hiking", "Snowshoing")
df_activities["activity"] = df_activities["activity"].replace("Ciaspolata", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Wandern", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Speedhiking", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Ski hike", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Ski hiking", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Metsäkevennys", "Forest")
df_activities["activity"] = df_activities["activity"].replace("Solo hiking", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Patikka (hiking)", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Langlaufen", "Cross-country skiing")
df_activities["activity"] = df_activities["activity"].replace("Staying in a holiday home", "Staying in a summer cottage, farm, holiday home")
df_activities["activity"] = df_activities["activity"].replace("Staying overnight in a summer cottage, farm, holiday home", "Staying in a summer cottage, farm, holiday home")
df_activities["activity"] = df_activities["activity"].replace("Traveling or staying in a caravan or camper", "Travelling or staying in a caravan or camper")
df_activities["activity"] = df_activities["activity"].replace("Sup boating", "SUP boating")
df_activities["activity"] = df_activities["activity"].replace("Nature observation (e.g. rafting)", "Nature observation")
df_activities["activity"] = df_activities["activity"].replace("Picking herbs", "Picking herbs, flowers or ornamental plants in nature")
df_activities["activity"] = df_activities["activity"].replace("Staying in a lodge, etc. in natural terrain", "Staying overnight in a tent, lodge, etc. in natural terrain")
df_activities["activity"] = df_activities["activity"].replace("Staying in a lodge", "Staying overnight in a tent, lodge, etc. in natural terrain")
df_activities["activity"] = df_activities["activity"].replace("Staying overnight in a tent", "Staying overnight in a tent, lodge, etc. in natural terrain")
df_activities["activity"] = df_activities["activity"].replace("Staying overnight in a lodge, etc. in natural terrain", "Staying overnight in a tent, lodge, etc. in natural terrain")
df_activities["activity"] = df_activities["activity"].replace("Flying a kite, drone or drone", "Flying a drone")
df_activities["activity"] = df_activities["activity"].replace("Camp", "Camping")
df_activities["activity"] = df_activities["activity"].replace("Sleighing", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Hands-free hiking with a dog", "Dog walking")
df_activities["activity"] = df_activities["activity"].replace("Wild camping", "Camping")
df_activities["activity"] = df_activities["activity"].replace("Winter camping", "Camping")
df_activities["activity"] = df_activities["activity"].replace("Staying in a farm, holiday home", "Staying in a summer cottage, farm, holiday home")
df_activities["activity"] = df_activities["activity"].replace("Staying in a summer cottage", "Staying in a summer cottage, farm, holiday home")
df_activities["activity"] = df_activities["activity"].replace("Scout excursion, camp or nature camp", "Scout excursion")
df_activities["activity"] = df_activities["activity"].replace("Diving, scuba diving", "Diving")
df_activities["activity"] = df_activities["activity"].replace("Night hiking or nature observation", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Night hiking", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Lumikenkäretki", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Night hiking or night trekking", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Night hiking or night exploration in forest", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Night hiking or trekking", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Sleigh ride", "Sledding")
df_activities["activity"] = df_activities["activity"].replace("Fuelwood procurement", "Fuelwood procurement, collection of twigs, branches and trees for domestic use")
df_activities["activity"] = df_activities["activity"].replace("Picking flowers", "Picking herbs, flowers or ornamental plants in nature")
df_activities["activity"] = df_activities["activity"].replace("Mountain climbing", "Climbing")
df_activities["activity"] = df_activities["activity"].replace("Staying in a caravan or camper", "Travelling or staying in a caravan or camper")
df_activities["activity"] = df_activities["activity"].replace("Snowmobile", "Snowmobiling")
df_activities["activity"] = df_activities["activity"].replace("Snowmobile riding", "Snowmobiling")
df_activities["activity"] = df_activities["activity"].replace("Motor sledge racing", "Snowmobiling")
df_activities["activity"] = df_activities["activity"].replace("Fishing with net", "Fishing")
df_activities["activity"] = df_activities["activity"].replace("Fish handling", "Fishing")
df_activities["activity"] = df_activities["activity"].replace("Fishing with trap", "Fishing")
df_activities["activity"] = df_activities["activity"].replace("Tenkara fishing", "Fishing")
df_activities["activity"] = df_activities["activity"].replace("Picking mushrooms", "Mushroom picking")
df_activities["activity"] = df_activities["activity"].replace("Ski touring", "Skiing")
df_activities["activity"] = df_activities["activity"].replace("Freeskiing", "Skiing")
df_activities["activity"] = df_activities["activity"].replace("Road biking", "Cycling")
df_activities["activity"] = df_activities["activity"].replace("Sunbathing or lounging on the beach", "Sunbathing")
df_activities["activity"] = df_activities["activity"].replace("Under ice freediving", "Diving")
df_activities["activity"] = df_activities["activity"].replace("Rib boating", "Boating")
df_activities["activity"] = df_activities["activity"].replace("Boat trip", "Boating")
df_activities["activity"] = df_activities["activity"].replace("Paddling", "Boating")
df_activities["activity"] = df_activities["activity"].replace("Diving, scuba diving, snorkelling", "Diving")
df_activities["activity"] = df_activities["activity"].replace("Laskettelua", "Downhill skiing")
df_activities["activity"] = df_activities["activity"].replace("Photography", "Nature photography")
df_activities["activity"] = df_activities["activity"].replace("Wildlife observation", "Nature observation")

In [114]:
# Update activity type to new_activity for those are not in the LVVI3 activity list
df_activities.loc[df_activities["activity"] == "Gold panning", "type"] = "new_activity"
df_activities.loc[df_activities["activity"] == "Bikepacking", "type"] = "new_activity"
df_activities.loc[df_activities["activity"] == "Bushcraft", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Outdoor sauna", "Sauna")
df_activities["activity"] = df_activities["activity"].replace("Relaxing in a sauna", "Sauna")
df_activities["activity"] = df_activities["activity"].replace("Sauna traditional finlandesa", "Sauna")
df_activities["activity"] = df_activities["activity"].replace("Sauna traditional finlandese", "Sauna")
df_activities.loc[df_activities["activity"] == "Sauna", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Kissing reindeers", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer herding", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Going to a reindeer reserve", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Rendiersafari", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Visiting reindeer farm", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Visiting a reindeer farm", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer farm visit", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer farm tour", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Meeting reindeer", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer ride", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer riding", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer sleigh ride", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer sledding", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer sled", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Caring for reindeer", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Reindeer feeding", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Feeding reindeer", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Feeding reindeers", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Feeding the reindeer", "Reindeer safari")
df_activities["activity"] = df_activities["activity"].replace("Riding reindeer", "Reindeer safari")
df_activities.loc[df_activities["activity"] == "Reindeer safari", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Dog sledding", "Husky safari")
df_activities["activity"] = df_activities["activity"].replace("Husky ride", "Husky safari")
df_activities["activity"] = df_activities["activity"].replace("Husky rides", "Husky safari")
df_activities["activity"] = df_activities["activity"].replace("Husky sledding", "Husky safari")
df_activities["activity"] = df_activities["activity"].replace("Husky-wandertouren", "Husky safari")
df_activities.loc[df_activities["activity"] == "Husky safari", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Ice fishing", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Backrafting", "Packrafting")
df_activities.loc[df_activities["activity"] == "Packrafting", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Snowkite", "Snowkiting")
df_activities.loc[df_activities["activity"] == "Snowkiting", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Electric snowmobile safaris", "Snowmobile safari")
df_activities["activity"] = df_activities["activity"].replace("Electric snowmobile safari", "Snowmobile safari")
df_activities.loc[df_activities["activity"] == "Snowmobile safari", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Fatbike-tours", "Fatbiking")
df_activities["activity"] = df_activities["activity"].replace("Fat bike", "Fatbiking")
df_activities["activity"] = df_activities["activity"].replace("Fat biking", "Fatbiking")
df_activities.loc[df_activities["activity"] == "Fatbiking", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Paragliding", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Heading out to see northern lights", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora boreal viewing", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora hunting", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Stargazing or viewing auroras", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora hunt", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Northern lights photography tour", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora boreale", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Watching northern lights", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora viewing", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora borealis viewing", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Northern lights hunting", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Northern lights viewing", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Northern lights expedition", "Northern lights watching")
df_activities["activity"] = df_activities["activity"].replace("Aurora watching", "Northern lights watching")
df_activities.loc[df_activities["activity"] == "Northern lights watching", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Snowsurfing", "type"] = "new_activity"
df_activities.loc[df_activities["activity"] == "Ice climbing", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Ice swimming", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Motorbike riding", "Motorbike riding")
df_activities.loc[df_activities["activity"] == "Motorbike riding", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Grilling", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Kitesurfing", "type"] = "new_activity"

df_activities.loc[df_activities["activity"] == "Slacklining", "type"] = "new_activity"

df_activities["activity"] = df_activities["activity"].replace("Acro yoga", "Acroyoga")
df_activities.loc[df_activities["activity"] == "Acroyoga", "type"] = "new_activity"


In [115]:
# Rename activities and label those relevant to LVVI3 activities as survey_activity
df_activities["activity"] = df_activities["activity"].replace("Kävelylle (walking)", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Talvivaellus", "Hiking")
df_activities["activity"] = df_activities["activity"].replace("Winter hiking", "Hiking")
df_activities.loc[df_activities["activity"] == "Hiking", "type"] = "survey_activity"

df_activities["activity"] = df_activities["activity"].replace("Telemark skiing", "Skiing")
df_activities["activity"] = df_activities["activity"].replace("Freestyle skiing", "Skiing")
df_activities["activity"] = df_activities["activity"].replace("Heli-mini-skiing", "Skiing")
df_activities.loc[df_activities["activity"] == "Skiing", "type"] = "survey_activity"

df_activities["activity"] = df_activities["activity"].replace("Kullanhuuhdonta", "Gold panning")
df_activities["activity"] = df_activities["activity"].replace("Gold hunting", "Gold panning")
df_activities["activity"] = df_activities["activity"].replace("Gold prospecting", "Gold panning")

df_activities["activity"] = df_activities["activity"].replace("Fińska sauna", "Sauna")

df_activities["activity"] = df_activities["activity"].replace("River swimming", "Swimming")
df_activities.loc[df_activities["activity"] == "Swimming", "type"] = "survey_activity"

df_activities["activity"] = df_activities["activity"].replace("Sleigh riding", "Sledding")
df_activities.loc[df_activities["activity"] == "Sledding", "type"] = "survey_activity"

df_activities.loc[df_activities["activity"] == "Dog riding", "type"] = "survey_activity"

In [116]:
df_activities_survey = df_activities[df_activities['type'] == 'survey_activity']
ls_activities_survey = df_activities_survey['activity'].unique().tolist()
ls_activities_survey_not_matching = [activity for activity in ls_activities_survey if activity not in ls_activities_post]
len(ls_activities_survey_not_matching)

2

In [117]:
df_activities_new = df_activities[df_activities['type'] == 'new_activity']
ls_activities_new = df_activities_new['activity'].unique().tolist()
ls_activities_new_matching = [activity for activity in ls_activities_new if activity in ls_activities_post]
# Update activity type to survey_activity for those are in the LVVI3 activity list
for act in ls_activities_new_matching:
    df_activities.loc[df_activities["activity"] == act, "type"] = "survey_activity"

In [118]:
df_activities_new = df_activities[df_activities['type'] == 'new_activity']
ls_activities_new = df_activities_new['activity'].unique().tolist()
len(ls_activities_new)

220

In [119]:
ls_activities_all = df_activities_post_origin.sub_activity.tolist()
ls_activities_to_remove = [act for act in ls_activities_new if act not in ls_activities_all]

# # [item for item in ls if "northern" in item.lower()]

df_activities = df_activities[~df_activities['activity'].isin(ls_activities_to_remove)]

In [120]:
df_activities_new = df_activities[df_activities['type'] == 'new_activity']
ls_activities_new = df_activities_new['activity'].unique().tolist()
ls_activities_all = df_activities_post_origin.sub_activity.tolist()
ls_activities_to_remove = [act for act in ls_activities_new if act not in ls_activities_all]
ls_activities_to_remove

[]

## Map sub_activities to activities

In [122]:
# Create a mapping: sub_activity → activity
sub_to_activity = df_activities_post_origin.set_index('sub_activity')['activity'].to_dict()

# Rename the activity column to sub_activity
df_activities = df_activities.rename(columns={"activity": "sub_activity"})
# Map the 'sub_activity' column to get the corresponding 'activity'
df_activities['activity'] = df_activities['sub_activity'].map(sub_to_activity)

# Update activity type
df_activities = df_activities.drop(columns=['type'])
df_activities = df_activities.merge(df_activities_post_origin[['structural_activity', 'sub_activity', 'type']], on='sub_activity', how='left')
df_activities = df_activities[["video_id", "activity", "sub_activity", "type", "structural_activity"]]

In [123]:
df_activities

,video_id,activity,sub_activity,type,structural_activity
0,-0-sLChwN2w,Walking,Hiking,survey_activity,Leisure
1,-0-sLChwN2w,Orienteering,Orienteering,survey_activity,Adventure
2,-0-sLChwN2w,"Staying overnight in a tent, lodge, etc. in na...","Staying overnight in a tent, lodge, etc. in na...",survey_activity,Nature Immersion
3,-0-sLChwN2w,Camping,Camping,survey_activity,Nature Immersion
4,-0-sLChwN2w,Nature photography,Nature photography,survey_activity,Nature Appreciation
...,...,...,...,...,...
8093,_zR8UTFSE3c,Snowmobiling,Snowmobiling,survey_activity,Adventure
8094,_zR8UTFSE3c,Husky safari,Husky safari,new_activity,Organized Experience
8095,_ZyNsg4Dwns,Camping,Camping,survey_activity,Nature Immersion
8096,_ZyNsg4Dwns,"Staying overnight in a tent, lodge, etc. in na...","Staying overnight in a tent, lodge, etc. in na...",survey_activity,Nature Immersion


## Save cleaned data

In [ ]:
# save cleaned activity dataframe as the csv file
# df_activities.to_csv("../../../data/_archive/old_llm_subactivity/youtube/YouTube_inferenced.csv", index=False)
df_activities = pd.read_csv("../../../data/_archive/old_llm_subactivity/youtube/YouTube_inferenced.csv")

# Clean accompany types

## Read inferenced accompany types from JSON files

In [ ]:
# Extract accompany types from all JSON files and create a DataFrame
def extract_accompany(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)

    video_id = data.get("video_id")
    rows = []

    for shot in data.get("shots", []):
        accompany_type = shot.get("accompany_type", "").strip()
        rows.append({
            "video_id": video_id,
            "accompany_type": accompany_type,
        })

    return rows

all_rows = []
json_folder = Path("../../../data/_archive/old_llm_subactivity/youtube/accompany_inference_json_output")

for file in json_folder.glob("*.json"):
    all_rows.extend(extract_accompany(file))

df_accompany = pd.DataFrame(all_rows)
df_accompany = df_accompany[df_accompany['video_id'].isin(ls_video_id)]
df_accompany = df_accompany.drop_duplicates().reset_index(drop=True)
df_accompany.shape

In [ ]:
# Select one accompany type for the video
priority = [
    "With family",
    "With work colleagues/business guests",
    "With school class/student group",
    "With club, association",
    "With other group",
    "With friends",
    "Alone"
]

df_accompany['accompany_type'] = pd.Categorical(
    df_accompany['accompany_type'],
    categories=priority,
    ordered=True
)

df_accompany = (
    df_accompany
    .dropna(subset=['accompany_type'])           # drop empty
    .sort_values('accompany_type')              # sorts by priority
    .groupby('video_id', as_index=False)
    .first()                                     # take highest priority accompany_type
)

In [ ]:
df_activities = df_activities.merge(df_accompany, on='video_id', how='left')

## Save cleaned data

In [ ]:
# save cleaned activity dataframe as the csv file
# df_activities.to_csv("../../../data/_archive/old_llm_subactivity/youtube/YouTube_inferenced.csv", index=False)
df_activities = pd.read_csv("../../../data/_archive/old_llm_subactivity/youtube/YouTube_inferenced.csv")